In [2]:
# Install dependencies
%pip install Anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# Create an API client
from anthropic import Anthropic
from anthropic.types import ThinkingConfigDisabledParam

client = Anthropic()
model = "claude-sonnet-4-5" # Needed for prefill support

In [5]:
# Helper functions
def add_user_message(messages: list[dict], text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list[dict], text: str):
    messages.append({"role": "assistant", "content": text})    


def chat(messages: list[dict], 
         system_prompt:str|None=None, 
         temperature:float=1.0,
         stop_sequences:list[str]|None=None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        params["thinking"] = {"type": "disabled"} # Thinking is not supported when using assistant message prefill

    return client.messages.create(**params).content[0].text   


In [6]:
# Math question example (without system prompt)
messages = []

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
answer = chat(messages)
print(answer)


To solve 5x + 3 = 2 for x, isolate x using these steps:

**Step 1:** Subtract 3 from both sides
- 5x + 3 - 3 = 2 - 3
- 5x = -1

**Step 2:** Divide both sides by 5
- 5x/5 = -1/5
- x = -1/5

**Answer: x = -1/5** (or -0.2 in decimal form)

**Check:** 5(-1/5) + 3 = -1 + 3 = 2 ✓


In [7]:
# Math question example (with Math tutor system prompt)
messages = []
system_prompt = """
You are a patient math tutor.
Do not directly give the answer to a student's question.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
answer = chat(messages, system_prompt=system_prompt)
print(answer)

Great question! Let's work through this together step by step.

When solving an equation like 5x + 3 = 2, our goal is to get x by itself on one side. 

**First step**: What do you notice is "with" the 5x on the left side of the equation? What operation could we perform to both sides to remove it?


In [8]:
# Concise coder example
messages = []
system_prompt = """
You are a python engineer who writes concise and efficient code.
Only write the code, do not include any explanations or comments.
Do not give options, only the most efficient solution.
"""

add_user_message(messages, "Write a Python function that checks a string for duplicate characters")
answer = chat(messages, system_prompt=system_prompt)
print(answer)

```python
def has_duplicate_characters(s: str) -> bool:
    return len(s) != len(set(s))
```


In [9]:
# Generate two similar movie ideas by using a low temperature
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=0.0)
print(answer)

A lonely astronaut discovers that the mysterious signal she's been chasing across the galaxy is actually her own distress call from the future.


In [10]:
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=0.0)
print(answer)

A time-traveling food critic accidentally prevents the invention of cooking and must recreate humanity's first cooked meal before civilization collapses into chaos.


In [11]:
# More creative movie ideas by using a higher temperature
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=1.0)
print(answer)

A time-traveling food critic accidentally prevents the invention of cooking and must recreate humanity's first meal before civilization collapses into chaos.


In [12]:
# Response streaming example
messages = []
add_user_message(messages, "Write a one sentence description of a fake database")

stream = client.messages.create(
        model=model,    
        max_tokens=1000,
        messages=messages,
        stream=True)

for chunk in stream:
    print(chunk)

RawMessageStartEvent(message=Message(id='msg_01KVLrbjbnQnfg7WCZhyA3x6', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=6, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fake database is a sim', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated,', type='text_delta'), index=0, type='content_block_delt

In [13]:
# Streaming with chunked text deltas
messages = []
add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
        model=model,    
        max_tokens=1000,
        messages=messages) as stream:
    for chunk in stream.text_stream:
        print(chunk)

    print("Streaming Done!")
    print("And we can access the full response content here:")
    print(stream.get_final_message().content[0].text)

A fake database is a sim
ulated or mock data storage system that mimics the structure and behavior of a real database but
 contains fabricated, placeholder, or test data instead of actual production
 information.
Streaming Done!
And we can access the full response content here:
A fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated, placeholder, or test data instead of actual production information.


In [14]:
# Event bridge rules example showing structured output
# Note this technique doesn't work for newer models that don't support assistant message prefill (e.g. sonnet-4-6+)
messages = []
add_user_message(messages, "Generate a very short event bridge rule in JSON.")
add_assistant_message(messages, "```json")
answer = chat(messages, stop_sequences=["```"])
print(answer)


{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}



In [15]:
# Structured output via Pydantic - the modern replacement for prefill on Sonnet 4.6+
# Uses client.messages.parse() with a Pydantic model as output_format.
from pydantic import BaseModel

class EventBridgeRule(BaseModel):
    source: list[str]
    detail_type: list[str]
    state: list[str]

messages = []
add_user_message(messages, "Generate a very short event bridge rule in JSON.");

response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    output_format=EventBridgeRule,
)

print(response.parsed_output)
print(type(response.parsed_output))

source=['aws.ec2'] detail_type=['EC2 Instance State-change Notification'] state=['running', 'stopped']
<class '__main__.EventBridgeRule'>


In [16]:
# Exercise: Generate three different sample AWS CLI commands using a system prompt to specify the format. 
# Use stop sequences to ensure only the command is returned without any additional explanation.
messages = []
prompt = """
Generate three different sample AWS CLI command. Each should be very short
"""

add_user_message(messages, prompt)  
add_assistant_message(messages, "Here are all 3 commands in a single block without any comments:\n ```bash")
answer = chat(messages, stop_sequences=["```"])
print(answer)


aws s3 ls

aws ec2 describe-instances

aws lambda list-functions



In [18]:
# Structured output via Pydantic - the modern replacement for prefill on Sonnet 4.6+
# Uses client.messages.parse() with a Pydantic model as output_format.
from pydantic import BaseModel

class AWSCLICommand(BaseModel):
    commands: list[str]

messages = []
add_user_message(messages, "Generate three different sample AWS CLI command. Each should be very short");

response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    output_format=AWSCLICommand,
)

print(response.parsed_output)
print(type(response.parsed_output))

commands=['aws s3 ls', 'aws ec2 describe-instances', 'aws iam list-users']
<class '__main__.AWSCLICommand'>
